In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce

In [0]:
%sql
USE CATALOG ecommerce;
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
%sql
USE SCHEMA bronze;
CREATE VOLUME IF NOT EXISTS landing

In [0]:
#Setando minhas variáveis para facilitar a chamada
catalogo = "ecommerce"
bronze_schema = "bronze"

In [0]:
from pyspark.sql import functions as F

#Função para ler os arquivos e criar as tabelas
try:
    def ingest_csv(nome_arquivo, nome_tabela):
        # Caminho da landing (Dados totalmente crus)
        landing_path = f"/Volumes/ecommerce/default/landing/{nome_arquivo}"

        # Lê o CSV no schema Bronze
        df = spark.read.csv(landing_path, header=True, inferSchema=True)

        # Impede ingestão de arquivos vazios
        if df.count() == 0:
            raise ValueError(f"O Arquivo {nome_arquivo} está vazio")

        # Adiciona o timestamp da ingestão
        df = df.withColumn("timestamp_ingestion", F.current_timestamp())

        # Salva como tabela Delta no schema Bronze
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{bronze_schema}.{nome_tabela}")

        print(f"Tabela {nome_tabela} criada com sucesso")

except Exception as e:
    print(f"Ocorreu um erro: {e}")

In [0]:
# Dicionário com os nomes dos arquivos e nomes das suas respectivas tabelas
tabelas = {
    "olist_customers_dataset.csv":                "tb_customers",
    "olist_geolocation_dataset.csv":              "tb_geolocalizacao",
    "olist_order_items_dataset.csv":              "tb_order_items",
    "olist_order_payments_dataset.csv":           "tb_order_payments",
    "olist_order_reviews_dataset.csv":            "tb_order_reviews",
    "olist_orders_dataset.csv":                   "tb_orders",
    "olist_products_dataset.csv":                 "tb_products",
    "olist_sellers_dataset.csv":                  "tb_sellers",
    "product_category_name_translation.csv":      "tb_product_category_name_translation",
}

# Chama a função para cada um dos arquivos
for arquivo, tabela in tabelas.items():
    ingest_csv(arquivo, tabela)

Tabela tb_customers criada com sucesso
Tabela tb_geolocalizacao criada com sucesso
Tabela tb_order_items criada com sucesso
Tabela tb_order_payments criada com sucesso
Tabela tb_order_reviews criada com sucesso
Tabela tb_orders criada com sucesso
Tabela tb_products criada com sucesso
Tabela tb_sellers criada com sucesso
Tabela tb_product_category_name_translation criada com sucesso


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import requests

# 1. Ler tabela de pedidos para pegar o período
df_pedidos = spark.table(f"{catalogo}.{bronze_schema}.tb_orders")

# 2. Obter data mínima e máxima
datas = df_pedidos.select(
    F.min("order_purchase_timestamp").alias("data_inicio"),
    F.max("order_purchase_timestamp").alias("data_fim")
).collect()[0]

data_inicio = datas["data_inicio"]
data_fim = datas["data_fim"]

# 3. Garantir tipo datetime
if isinstance(data_inicio, str):
    data_inicio = datetime.strptime(data_inicio, "%Y-%m-%d")
if isinstance(data_fim, str):
    data_fim = datetime.strptime(data_fim, "%Y-%m-%d")

# 4. Formatar para padrão da API (MM-DD-YYYY)
data_inicio_formatada = data_inicio.strftime("%m-%d-%Y")
data_fim_formatada = data_fim.strftime("%m-%d-%Y")

print("Período:", data_inicio_formatada, "até", data_fim_formatada)

# 5. Montar URL
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# 6. Requisição
response = requests.get(url)
data_json = response.json()
valores = data_json.get("value", [])

# 7. Verificar retorno
if not valores:
    print("Nenhum dado retornado pela API")

else:
    # 8. Criar DataFrame
    df_cotacao = spark.createDataFrame(valores)

    # 9. 🔥 Converter timestamp para DATE (ESSENCIAL pro pipeline)
    df_cotacao = df_cotacao.withColumn(
        "data_cotacao",
        F.to_date("dataHoraCotacao")
    ).drop("dataHoraCotacao")

    # Apenas para validar
    #df_cotacao.display()

    # 10. Salvar na Bronze
    (df_cotacao.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalogo}.{bronze_schema}.tb_cotacao_dolar")
    )

    print(f"✓ tb_cotacao_dolar ({df_cotacao.count()} linhas)")

Período: 09-04-2016 até 10-17-2018


cotacaoCompra,data_cotacao
3.2715,2016-09-05
3.2446,2016-09-06
3.1928,2016-09-08
3.2632,2016-09-09
3.2848,2016-09-12
3.2966,2016-09-13
3.3256,2016-09-14
3.332,2016-09-15
3.2998,2016-09-16
3.263,2016-09-19


✓ tb_cotacao_dolar (530 linhas)
